# Text HMTL V4 训练 (Colab)

**目标**: 7/4/3 类准确率都要好

**V3问题**: 7类权重太低导致7类准确率下降

**V4改进**: 平衡各类损失权重

## 1. 环境检查

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

!pip install transformers -q

## 2. 挂载 Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

DRIVE_PATH = "/content/drive/MyDrive"
TRAIN_DATA = f"{DRIVE_PATH}/training_set_hmtl_expanded.json"
MODEL_SAVE_PATH = f"{DRIVE_PATH}/text_hmtl_v4_best.pt"

print(f"训练数据存在: {os.path.exists(TRAIN_DATA)}")

## 3. 定义模型

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import BertModel


class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction
    
    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        p_t = torch.exp(-ce_loss)
        focal_weight = (1 - p_t) ** self.gamma
        
        if self.alpha is not None:
            if isinstance(self.alpha, torch.Tensor):
                alpha_t = self.alpha[targets]
                focal_weight = alpha_t * focal_weight
        
        focal_loss = focal_weight * ce_loss
        
        if self.reduction == 'mean':
            return focal_loss.mean()
        return focal_loss


class TextHMTLModelV4(nn.Module):
    """
    Text HMTL V4 - 7/4/3 类都要好
    """
    
    def __init__(self, bert_model_name='bert-base-chinese', dropout=0.3):
        super().__init__()
        
        self.bert = BertModel.from_pretrained(bert_model_name)
        hidden_size = self.bert.config.hidden_size
        
        # 7类分类头 (重要)
        self.classifier_7 = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 384),
            nn.ReLU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(384, 7)
        )
        
        # 4类分类头 (重要)
        self.classifier_4 = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 256),
            nn.ReLU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(256, 4)
        )
        
        # 3类分类头
        self.classifier_3 = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 128),
            nn.ReLU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(128, 3)
        )
        
        # Arousal/Valence
        self.arousal_head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 128),
            nn.ReLU(),
            nn.Linear(128, 1),
            nn.Sigmoid()
        )
        
        self.valence_head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 128),
            nn.ReLU(),
            nn.Linear(128, 1),
            nn.Tanh()
        )
    
    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :]
        
        return {
            'label_7_logits': self.classifier_7(cls_output),
            'label_4_logits': self.classifier_4(cls_output),
            'label_3_logits': self.classifier_3(cls_output),
            'arousal': self.arousal_head(cls_output).squeeze(-1),
            'valence': self.valence_head(cls_output).squeeze(-1)
        }


print("模型定义完成!")

## 4. 数据集和损失函数

In [ ]:
import json
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer


class TextEmotionDataset(Dataset):
    def __init__(self, data_path, tokenizer, max_length=128):
        with open(data_path, 'r', encoding='utf-8') as f:
            self.data = json.load(f)
        self.tokenizer = tokenizer
        self.max_length = max_length
        print(f"加载数据: {len(self.data)} 条")
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        item = self.data[idx]
        
        encoding = self.tokenizer(
            item['text'],
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        label_7 = item.get('label_7', 6)
        
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'label_7': torch.tensor(label_7, dtype=torch.long),
            'label_4': torch.tensor(item['label_4'], dtype=torch.long),
            'label_3': torch.tensor(item['label_3'], dtype=torch.long),
            'arousal': torch.tensor(item['arousal'], dtype=torch.float),
            'valence': torch.tensor(item['valence'], dtype=torch.float),
        }


class HMTLLossV4(nn.Module):
    """
    V4 Loss: 7/4/3 类平衡
    """
    def __init__(self, class_weights_7=None, class_weights_4=None):
        super().__init__()
        
        # 7类和4类都用 Focal Loss
        self.loss_7 = FocalLoss(alpha=class_weights_7, gamma=2.0)
        self.loss_4 = FocalLoss(alpha=class_weights_4, gamma=2.0)
        self.loss_3 = nn.CrossEntropyLoss()
        self.mse_loss = nn.MSELoss()
    
    def forward(self, outputs, targets):
        # 7类损失 (权重高)
        l7 = self.loss_7(outputs['label_7_logits'], targets['label_7'])
        
        # 4类损失 (权重高)
        l4 = self.loss_4(outputs['label_4_logits'], targets['label_4'])
        
        # 3类损失
        l3 = self.loss_3(outputs['label_3_logits'], targets['label_3'])
        
        # A/V 损失
        l_a = self.mse_loss(outputs['arousal'], targets['arousal'])
        l_v = self.mse_loss(outputs['valence'], targets['valence'])
        
        # V4: 7类和4类权重相同，都是1.2
        total = 1.2 * l7 + 1.2 * l4 + 0.8 * l3 + 0.5 * l_a + 0.3 * l_v
        
        return {
            'total_loss': total,
            'loss_7': l7.item(),
            'loss_4': l4.item(),
            'loss_3': l3.item(),
        }


print("数据集和损失函数定义完成!")

## 5. 加载数据

In [ ]:
from sklearn.model_selection import train_test_split

tokenizer = BertTokenizer.from_pretrained('bert-base-chinese')

with open(TRAIN_DATA, 'r', encoding='utf-8') as f:
    all_data = json.load(f)

print(f"总样本数: {len(all_data)}")

# 统计7类分布
label_7_counts = {i: 0 for i in range(7)}
label_4_counts = {i: 0 for i in range(4)}
for item in all_data:
    label_7 = item.get('label_7', 6)
    label_7_counts[label_7] += 1
    label_4_counts[item['label_4']] += 1

EMOTION_7_NAMES = {0: '愤怒', 1: '焦虑', 2: '快乐', 3: '悲伤', 4: '失望', 5: '支持', 6: '平静'}
LABEL_4_NAMES = {0: '积极', 1: '激活消极', 2: '非激活消极', 3: '平静'}

print("\n7类分布:")
for i, name in EMOTION_7_NAMES.items():
    print(f"  [{i}] {name}: {label_7_counts[i]}")

print("\n4类分布:")
for i, name in LABEL_4_NAMES.items():
    print(f"  [{i}] {name}: {label_4_counts[i]}")

# 划分
train_data, val_data = train_test_split(all_data, test_size=0.15, random_state=42)

with open('/content/train_temp.json', 'w', encoding='utf-8') as f:
    json.dump(train_data, f, ensure_ascii=False)
with open('/content/val_temp.json', 'w', encoding='utf-8') as f:
    json.dump(val_data, f, ensure_ascii=False)

train_dataset = TextEmotionDataset('/content/train_temp.json', tokenizer)
val_dataset = TextEmotionDataset('/content/val_temp.json', tokenizer)

BATCH_SIZE = 16
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"\n训练集: {len(train_dataset)}, 验证集: {len(val_dataset)}")

## 6. 创建模型

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"设备: {device}")

model = TextHMTLModelV4().to(device)

# 7类权重
total_7 = sum(label_7_counts.values())
CLASS_WEIGHTS_7 = torch.tensor([
    total_7 / (7 * max(label_7_counts[i], 1)) for i in range(7)
]).to(device)
print(f"7类权重: {CLASS_WEIGHTS_7}")

# 4类权重
total_4 = sum(label_4_counts.values())
CLASS_WEIGHTS_4 = torch.tensor([
    total_4 / (4 * label_4_counts[i]) for i in range(4)
]).to(device)
print(f"4类权重: {CLASS_WEIGHTS_4}")

criterion = HMTLLossV4(class_weights_7=CLASS_WEIGHTS_7, class_weights_4=CLASS_WEIGHTS_4)

LEARNING_RATE = 2e-5

bert_params = list(model.bert.parameters())
head_params = (
    list(model.classifier_7.parameters()) +
    list(model.classifier_4.parameters()) +
    list(model.classifier_3.parameters()) +
    list(model.arousal_head.parameters()) +
    list(model.valence_head.parameters())
)

optimizer = torch.optim.AdamW([
    {'params': bert_params, 'lr': LEARNING_RATE},
    {'params': head_params, 'lr': LEARNING_RATE * 5}
])

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=15)

print("模型创建完成!")

## 7. 训练

In [ ]:
from tqdm import tqdm

def evaluate(model, dataloader, device):
    model.eval()
    correct_7, correct_4, correct_3 = 0, 0, 0
    total = 0
    
    class_correct_7 = {i: 0 for i in range(7)}
    class_total_7 = {i: 0 for i in range(7)}
    class_correct_4 = {i: 0 for i in range(4)}
    class_total_4 = {i: 0 for i in range(4)}
    
    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            targets = {k: v.to(device) for k, v in batch.items() 
                      if k not in ['input_ids', 'attention_mask']}
            
            outputs = model(input_ids, attention_mask)
            
            pred_7 = outputs['label_7_logits'].argmax(dim=1)
            pred_4 = outputs['label_4_logits'].argmax(dim=1)
            pred_3 = outputs['label_3_logits'].argmax(dim=1)
            
            correct_7 += (pred_7 == targets['label_7']).sum().item()
            correct_4 += (pred_4 == targets['label_4']).sum().item()
            correct_3 += (pred_3 == targets['label_3']).sum().item()
            total += targets['label_7'].size(0)
            
            for i in range(len(pred_7)):
                t7 = targets['label_7'][i].item()
                t4 = targets['label_4'][i].item()
                class_total_7[t7] += 1
                class_total_4[t4] += 1
                if pred_7[i].item() == t7:
                    class_correct_7[t7] += 1
                if pred_4[i].item() == t4:
                    class_correct_4[t4] += 1
    
    return {
        'acc_7': correct_7 / total,
        'acc_4': correct_4 / total,
        'acc_3': correct_3 / total,
        'class_acc_7': {k: class_correct_7[k]/class_total_7[k] if class_total_7[k] > 0 else 0 for k in range(7)},
        'class_acc_4': {k: class_correct_4[k]/class_total_4[k] if class_total_4[k] > 0 else 0 for k in range(4)}
    }


NUM_EPOCHS = 15
best_score = 0  # 综合分数

print("="*60)
print("开始训练 Text HMTL V4")
print(f"目标: 7/4/3 类准确率都要好")
print("="*60)

for epoch in range(NUM_EPOCHS):
    model.train()
    epoch_loss = 0
    
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}")
    for batch in pbar:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        targets = {k: v.to(device) for k, v in batch.items() 
                  if k not in ['input_ids', 'attention_mask']}
        
        optimizer.zero_grad()
        outputs = model(input_ids, attention_mask)
        loss_dict = criterion(outputs, targets)
        
        loss_dict['total_loss'].backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        epoch_loss += loss_dict['total_loss'].item()
        pbar.set_postfix({'loss': f"{loss_dict['total_loss'].item():.4f}"})
    
    scheduler.step()
    
    # 评估
    eval_result = evaluate(model, val_loader, device)
    
    # 综合分数 = 7类*0.4 + 4类*0.4 + 3类*0.2
    score = eval_result['acc_7'] * 0.4 + eval_result['acc_4'] * 0.4 + eval_result['acc_3'] * 0.2
    
    print(f"\nEpoch {epoch+1}: Loss={epoch_loss/len(train_loader):.4f}")
    print(f"  7类: {eval_result['acc_7']*100:.2f}%, 4类: {eval_result['acc_4']*100:.2f}%, 3类: {eval_result['acc_3']*100:.2f}%")
    print(f"  综合分数: {score*100:.2f}%")
    
    # 保存最佳
    if score > best_score:
        best_score = score
        torch.save({
            'model_state_dict': model.state_dict(),
            'epoch': epoch,
            'acc_7': eval_result['acc_7'],
            'acc_4': eval_result['acc_4'],
            'acc_3': eval_result['acc_3'],
            'score': score,
        }, MODEL_SAVE_PATH)
        print(f"  -> 保存最佳模型!")

print("\n" + "="*60)
print(f"训练完成! 最佳综合分数: {best_score*100:.2f}%")
print(f"模型保存到: {MODEL_SAVE_PATH}")
print("="*60)

## 8. 测试

In [ ]:
checkpoint = torch.load(MODEL_SAVE_PATH)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

print(f"最佳模型:")
print(f"  7类: {checkpoint['acc_7']*100:.2f}%")
print(f"  4类: {checkpoint['acc_4']*100:.2f}%")
print(f"  3类: {checkpoint['acc_3']*100:.2f}%")

test_texts = [
    "太棒了！超级开心！",
    "气死我了！太差了！",
    "好担心明天的结果",
    "心里好难受...",
    "真让人失望",
    "加油，你可以的！",
    "还行吧，一般般",
]

print("\n测试样本:")
for text in test_texts:
    encoding = tokenizer(text, max_length=128, padding='max_length', 
                        truncation=True, return_tensors='pt')
    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)
    
    with torch.no_grad():
        outputs = model(input_ids, attention_mask)
    
    pred_7 = outputs['label_7_logits'].argmax(dim=1).item()
    pred_4 = outputs['label_4_logits'].argmax(dim=1).item()
    
    print(f"  {text}")
    print(f"    7类: {EMOTION_7_NAMES[pred_7]}, 4类: {LABEL_4_NAMES[pred_4]}")

## 9. 下载模型

In [ ]:
from google.colab import files

DRIVE_PATH = "/content/drive/MyDrive"
MODEL_SAVE_PATH = f"{DRIVE_PATH}/text_hmtl_v4_best.pt"

files.download(MODEL_SAVE_PATH)
print("下载完成!")